In [ ]:

#need some python libraries
import copy
import math
from random import Random
import numpy as np


#to setup a random number generator, we will specify a "seed" value
seed = 12345
myPRNG = Random(seed)

#to get a random number between 0 and 1, write call this:             myPRNG.random()
#to get a random number between lwrBnd and upprBnd, write call this:  myPRNG.uniform(lwrBnd,upprBnd)
#to get a random integer between lwrBnd and upprBnd, write call this: myPRNG.randint(lwrBnd,upprBnd)

lowerBound = -500  #bounds for Schwefel Function search space
upperBound = 500   #bounds for Schwefel Function search space

#you may change anything below this line that you wish too -----------------------------------------------------

#note: for the more experienced Python programmers, you might want to consider taking a more object-oriented approach to the PSO implementation, i.e.: a particle class with methods to initialize itself, and update its own velocity and position; a swarm class with a method to iterates through all particles to call update functions, etc.

#number of dimensions of problem
dimensions = 200

#number of particles in swarm
swarmSize = 100

maxIterations = 10000  # maximum number of iterations
noImprovementMaxIterations = 50  # number of iterations without improvement before stopping

#Schwefel function to evaluate a real-valued solution x
# note: the feasible space is an n-dimensional hypercube centered at the origin with side length = 2 * 500

def evaluate(x):
      val = 0
      d = len(x)
      for i in range(d):
            val = val + x[i]*math.sin(math.sqrt(abs(x[i])))

      val = 418.9829*d - val

      return val

# gunction that returns the global best position of the swarm
def getGlobalBest(pBest, pBestVal):
      gBest = pBest[0][:]  #initialize gBest to the first particle's best position
      gBestVal = pBestVal[0]  #initialize gBestVal to the first particle's best value

      for i in range(1, len(pBest)):
            if pBestVal[i] < gBestVal:
                  gBest = pBest[i][:]
                  gBestVal = pBestVal[i]

      return gBest, gBestVal

# PSO parameters
w = 0.729  # inertia weight
c1 = 1.5  # cognitive coefficient
c2 = 1.5  # social coefficient
velocityMax = 0.2 * (upperBound - lowerBound)  # maximum velocity

# velocity update function
def updateVelocities(vel, pos, pBest, gBest, w=w, c1=c1, c2=c2):  # corrected parameter w and added c2
    for i in range(len(vel)):
        for j in range(len(vel[i])):
            r1 = myPRNG.random()
            r2 = myPRNG.random()
            vel[i][j] = (w * vel[i][j]) + (c1 * r1 * (pBest[i][j] - pos[i][j])) + (c2 * r2 * (gBest[j] - pos[i][j]))
            # check if the velocity is within bounds
            if vel[i][j] < -velocityMax:
                vel[i][j] = -velocityMax
            elif vel[i][j] > velocityMax:
                vel[i][j] = velocityMax
    return vel

# position update function
def updatePositions(pos, vel):  # corrected function name
      for i in range(len(pos)):
            for j in range(len(pos[i])):
                  pos[i][j] += vel[i][j]
                  # check if the position is within bounds
                  if pos[i][j] < lowerBound:
                        pos[i][j] = lowerBound
                  elif pos[i][j] > upperBound:
                        pos[i][j] = upperBound
      return pos

#function to generate a summary of the swarm's current state
def summarizeSwarm(pos, vel, pBest, pBestVal):
    vel_array = np.array(vel)
    p_array = np.array(pBestVal)

    mean_vel = np.mean(vel_array)
    std_vel = np.std(vel_array)
    mean_p = np.mean(p_array)
    std_p = np.std(p_array)

    return mean_vel, std_vel, mean_p, std_p  # updated return statement to include mean_p and std_p

#the swarm will be represented as a list of positions, velocities, values, pbest, and pbest values

pos = [[] for _ in range(swarmSize)]      #position of particles -- will be a list of lists; e.g., for a 2D problem with 3 particles: [[17,4],[-100,2],[87,-1.2]]
vel = [[] for _ in range(swarmSize)]      #velocity of particles -- will be a list of lists similar to the "pos" object

#note: pos[0] and vel[0] provides the position and velocity of particle 0; pos[1] and vel[1] provides the position and velocity of particle 1; and so on.

curValue = [] #evaluation value of current position  -- will be a list of real values; curValue[0] provides the evaluation of particle 0 in it's current position
pBest = []    #particles' best historical position -- will be a list of lists: pBest[0] provides the position of particle 0's best historical position
pBestVal = [] #value of pbest position  -- will be a list of real values: pBestVal[0] provides the value of particle 0's pbest location


#initialize the swarm randomly
for i in range(swarmSize):
      for j in range(dimensions):
            pos[i].append(myPRNG.uniform(lowerBound,upperBound))    #assign random value between lower and upper bounds
            vel[i].append(myPRNG.uniform(-1,1))                     #assign random value between -1 and 1   --- maybe these are good bounds?  maybe not...
            # vel[i].append(myPRNG.uniform(-velocityMax, velocityMax))  # assign random velocity within the max bounds
      curValue.append(evaluate(pos[i]))   #evaluate the current position

pBest = pos[:]          # initialize pbest to the starting position
pBestVal = curValue[:]  # initialize pbest to the starting position

gBest , gBestVal = getGlobalBest(pBest, pBestVal)  # get the global best position and value

print(f"Initial Global Best Value = {gBestVal}")
print(f"Initial Global Best Position = {gBest}")

Initial Global Best Value = 76609.77350138675
Initial Global Best Position = [368.1968969759246, 343.728682740101, 398.69860109886895, 197.84687053067546, 119.48924924369544, 211.3461961414513, -71.0140870904765, -407.0347525630983, -94.69272045551304, 24.373821218506578, -329.87720669288035, -488.2884940356711, -284.01687257172455, 257.3142622912703, -408.6316756898193, -288.530257418651, 448.23180692081144, -115.97986874592846, -184.60429344513506, 168.20629747387238, -57.262406376492265, 389.1536033440775, 52.35864122904081, -40.010849691875876, -140.3774168357279, 32.21624389999863, 62.34562351873592, -320.2087725717141, 61.24692195063892, -30.504130841730955, -264.6592367104903, -268.58223346259524, 269.9052178342407, 22.988418892185337, 154.5993197510702, 103.23429396302527, -490.04701601072685, -48.688047467812964, -48.910703710065775, 192.76430066956084, 360.7071321185091, 272.8866704020387, 334.7605456859651, 271.2065165435723, 218.33789779428537, 159.35885763687065, 210.53199

In [6]:
vel_array = np.array(vel)
mean_vel = np.mean(vel_array)
print(f"Mean Velocity = {mean_vel}")


Mean Velocity = -0.0034106056896713896
